In [1]:
# ============================================================
# CELL 1 — Setup & GPU check
# Output: confirms GPU T4x2 availability, sets global config
# ============================================================
import numpy as np
import pandas as pd
import networkx as nx
import xgboost as xgb
import statsmodels.api as sm
import joblib
import json
import os
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                      capture_output=True, text=True).stdout)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete. XGBoost version:", xgb.__version__)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

Setup complete. XGBoost version: 3.2.0


In [2]:
# ============================================================
# CELL 2 — Dataset paths (EDIT to match whatever you attach)
# Output: none — just defines paths used by later cells
# ============================================================
# After adding "Data Science for Good: Kiva Crowdfunding" as a data source,
# Kaggle mounts it under /kaggle/input/<dataset-slug>/
# Replace <dataset-slug> below with the folder name shown in the Data panel.

KIVA_LOANS_PATH = "/kaggle/input/datasets/kiva/data-science-for-good-kiva-crowdfunding/kiva_loans.csv"
KIVA_MPI_PATH   = "/kaggle/input/datasets/kiva/data-science-for-good-kiva-crowdfunding/kiva_mpi_region_locations.csv"

# Swapping in a different real dataset only requires repointing these two
# paths — downstream cells just need columns roughly equivalent to:
# loan_amount, term_in_months, sector, country, region, partner_id, repayment_interval

In [3]:
# ============================================================
# CELL 3 — Load real Kiva data, clean, sample borrowers
# Output: `raw_df` — cleaned sample of REAL individual loan records
# ============================================================
loans = pd.read_csv(KIVA_LOANS_PATH)

keep_cols = ["id", "loan_amount", "sector", "country", "region",
             "term_in_months", "repayment_interval", "partner_id"]
loans = loans[keep_cols].dropna(subset=["loan_amount", "term_in_months", "partner_id"])

# Pull a larger real pool (400) than the final 100-150 export target, so
# groups/regions look realistic before trimming in Cell 9.
# swap this one line in Cell 3:
raw_df = loans.sample(n=min(2000, len(loans)), random_state=RANDOM_SEED).reset_index(drop=True)
raw_df = raw_df.rename(columns={"id": "loan_id", "term_in_months": "term_months"})

print(f"OUTPUT: raw_df loaded — shape={raw_df.shape}")
raw_df.head()

OUTPUT: raw_df loaded — shape=(2000, 8)


,loan_id,loan_amount,sector,country,region,term_months,repayment_interval,partner_id
0,952578,600.0,Health,Mexico,NaN,14.0,monthly,357.0
1,831874,475.0,Food,Guatemala,"San Martin Jilotepeque, Chimaltenango",14.0,monthly,97.0
2,1293258,3775.0,Retail,Burkina Faso,Ouagadougou Kilwin,6.0,irregular,398.0
3,1259338,475.0,Retail,Guatemala,"Ciudad Vieja,Sacatepequez",8.0,monthly,97.0
4,743144,500.0,Food,Zimbabwe,Chiredzi,8.0,irregular,305.0


In [4]:
# ============================================================
# CELL 4 — Assign borrowers to synthetic JLG groups + region buckets + savings
# FIXED: group formation is decoupled from partner_id (previously nested
# inside partner_id pools that average ~3.5 borrowers each, so 5-8 person
# groups were structurally impossible — 135 groups from 400 borrowers
# worked out to ~2.96 people/group). Region is also coarsened from raw
# Kiva region text (previously 292 near-unique values from 400 borrowers,
# meaning almost nobody could share a regional shock).
# Output: `borrowers_df`
# ============================================================
import hashlib

n_borrowers = len(raw_df)
GROUP_SIZE_RANGE = (5, 8)
N_REGION_BUCKETS_PER_COUNTRY = 4  # tune based on how many borrowers/country you end up with

borrowers_df = raw_df.copy()
borrowers_df["borrower_id"] = ["B" + str(i).zfill(4) for i in range(n_borrowers)]

# region_id: country + a deterministic hash-bucket of the raw region string,
# so borrowers in the same country land in a shared handful of regions
# instead of near-unique per-person values.
def make_region_id(row):
    country = str(row["country"])
    region_raw = str(row["region"])
    bucket = int(hashlib.md5(region_raw.encode()).hexdigest(), 16) % N_REGION_BUCKETS_PER_COUNTRY
    return f"{country}_R{bucket}"

borrowers_df["region_id"] = borrowers_df.apply(make_region_id, axis=1)

# group_id: chunk the FULL borrower pool directly into 5-8 person groups.
# partner_id is kept as its own independent column/edge type (branch/loan-
# officer proxy) — it is no longer used to construct groups.
idx = borrowers_df.index.tolist()
np.random.shuffle(idx)
group_records = []
group_counter = 0
i = 0
while i < len(idx):
    size = np.random.randint(*GROUP_SIZE_RANGE)
    chunk = idx[i:i + size]
    for row_idx in chunk:
        group_records.append((row_idx, f"G{group_counter:04d}"))
    group_counter += 1
    i += size

group_map = dict(group_records)
borrowers_df["group_id"] = borrowers_df.index.map(group_map)

borrowers_df["current_savings"] = (
    borrowers_df["loan_amount"] * np.random.uniform(0.05, 0.35, n_borrowers)
).round(2)

group_sizes = borrowers_df.groupby("group_id").size()
region_sizes = borrowers_df.groupby("region_id").size()

print(f"OUTPUT: borrowers_df — shape={borrowers_df.shape}, "
      f"{borrowers_df['group_id'].nunique()} groups (avg size {group_sizes.mean():.2f}), "
      f"{borrowers_df['region_id'].nunique()} regions (avg size {region_sizes.mean():.2f}), "
      f"{borrowers_df['partner_id'].nunique()} partners")
assert group_sizes.min() >= 2, "Group sizes fell below intended range — check trailing chunk logic"
borrowers_df.head()

OUTPUT: borrowers_df — shape=(2000, 12), 337 groups (avg size 5.93), 198 regions (avg size 10.10), 182 partners


,loan_id,loan_amount,sector,country,region,term_months,repayment_interval,partner_id,borrower_id,region_id,group_id,current_savings
0,952578,600.0,Health,Mexico,NaN,14.0,monthly,357.0,B0000,Mexico_R0,G0199,42.50
1,831874,475.0,Food,Guatemala,"San Martin Jilotepeque, Chimaltenango",14.0,monthly,97.0,B0001,Guatemala_R1,G0291,97.72
2,1293258,3775.0,Retail,Burkina Faso,Ouagadougou Kilwin,6.0,irregular,398.0,B0002,Burkina Faso_R2,G0076,265.32
3,1259338,475.0,Retail,Guatemala,"Ciudad Vieja,Sacatepequez",8.0,monthly,97.0,B0003,Guatemala_R1,G0171,137.80
4,743144,500.0,Food,Zimbabwe,Chiredzi,8.0,irregular,305.0,B0004,Zimbabwe_R1,G0251,60.06


In [5]:
# ============================================================
# CELL 5 — Build the multi-edge borrower relationship graph
# FIXED: now also builds `G_group`, a group-edges-ONLY graph, used by
# Cell 6 to compute neighbor_stress_fraction. Previously that fraction
# was computed on the fully merged graph, so it silently mixed in
# partner/region neighbors — the exact confound Model 3 is supposed to
# separate out (a "neighbor" wasn't guaranteed to be a real co-liability
# group-mate).
# Output: `G` (full multiplex graph, for GraphSAGE message passing +
#          edge_list.csv), `G_group` (group-only, for the contagion
#          feature), `group_edge_list.csv` saved
# ============================================================
G = nx.Graph()
G.add_nodes_from(borrowers_df["borrower_id"])
G_group = nx.Graph()
G_group.add_nodes_from(borrowers_df["borrower_id"])

EDGE_WEIGHTS = {"group": 1.0, "partner": 0.5, "region": 0.2}

def add_edges_for(df, key, weight, edge_type, group_graph=None):
    for _, sub in df.groupby(key):
        members = sub["borrower_id"].tolist()
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if G.has_edge(members[i], members[j]):
                    G[members[i]][members[j]]["weight"] += weight
                    G[members[i]][members[j]]["types"].append(edge_type)
                else:
                    G.add_edge(members[i], members[j], weight=weight, types=[edge_type])
                if group_graph is not None:
                    group_graph.add_edge(members[i], members[j])

add_edges_for(borrowers_df, "group_id",   EDGE_WEIGHTS["group"],   "group", group_graph=G_group)
add_edges_for(borrowers_df, "partner_id", EDGE_WEIGHTS["partner"], "partner")
add_edges_for(borrowers_df, "region_id",  EDGE_WEIGHTS["region"],  "region")

edge_list_df = pd.DataFrame(
    [(u, v, d["weight"], ",".join(d["types"])) for u, v, d in G.edges(data=True)],
    columns=["source", "target", "weight", "edge_types"]
)
edge_list_df.to_csv(OUTPUT_DIR + "edge_list.csv", index=False)

group_edge_list_df = pd.DataFrame(G_group.edges(), columns=["source", "target"])
group_edge_list_df.to_csv(OUTPUT_DIR + "group_edge_list.csv", index=False)

print(f"OUTPUT: graph built — {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (all types)")
print(f"OUTPUT: G_group — {G_group.number_of_edges()} group-only edges")
print(f"OUTPUT: edge_list.csv + group_edge_list.csv saved")

OUTPUT: graph built — 2000 nodes, 110868 edges (all types)
OUTPUT: G_group — 5055 group-only edges
OUTPUT: edge_list.csv + group_edge_list.csv saved


In [6]:
# ============================================================
# CELL 6 — Simulate stress evolution with a KNOWN injected mechanism
# FIXED (2 changes):
#  1. neighbor_stress_fraction now computed from G_group ONLY (group
#     edges), not the merged graph — previously silently mixed in
#     partner/region neighbors, the exact confound Model 3 is meant to
#     separate out.
#  2. States are now 3-way with RECOVERY: 0=healthy, 1=stressed,
#     2=defaulted (absorbing). Previously stress was permanent the
#     instant it hit, making own_lagged_risk near-tautological for
#     is_stressed — root cause of Model 1's hollow 0.971 train AUC.
# Output: `panel_df` — borrower x timestep panel
# ============================================================
T = 12
CONTAGION_BETA = 2.2
REGIONAL_SHOCK_PROB = 0.06
REGIONAL_SHOCK_BOOST = 3.0
RECOVERY_PROB = 0.25              # stressed -> healthy, per month
DEFAULT_PROB_IF_STRESSED = 0.08   # stressed -> defaulted, per month

loan_to_term = borrowers_df["loan_amount"] / borrowers_df["term_months"].clip(lower=1)
baseline_hazard = 0.02 + 0.04 * (loan_to_term / loan_to_term.max())
baseline_hazard += np.where(borrowers_df["repayment_interval"] != "Monthly", 0.02, 0.0)
borrowers_df["baseline_hazard"] = baseline_hazard.clip(0.01, 0.15)

group_neighbors = {n: list(G_group.neighbors(n)) for n in G_group.nodes}
regions = borrowers_df.set_index("borrower_id")["region_id"].to_dict()
region_members = borrowers_df.groupby("region_id")["borrower_id"].apply(list).to_dict()
hazard = borrowers_df.set_index("borrower_id")["baseline_hazard"].to_dict()

state = {b: 0 for b in borrowers_df["borrower_id"]}     # 0=healthy, 1=stressed, 2=defaulted
reason = {b: None for b in borrowers_df["borrower_id"]}

records = []
region_shock_active = {r: False for r in region_members}

for t in range(T):
    for r in region_members:
        region_shock_active[r] = np.random.rand() < REGIONAL_SHOCK_PROB

    prev_state = state.copy()
    for b in borrowers_df["borrower_id"]:
        cur = prev_state[b]
        nbrs = group_neighbors.get(b, [])
        neighbor_frac = (np.mean([1 if prev_state[n] in (1, 2) else 0 for n in nbrs]) if nbrs else 0.0)
        r = regions[b]

        if cur == 2:
            state[b] = 2  # defaulted stays defaulted

        elif cur == 1:
            roll = np.random.rand()
            if roll < RECOVERY_PROB:
                state[b] = 0
                reason[b] = None  # clears — a future re-onset gets a fresh reason
            elif roll < RECOVERY_PROB + DEFAULT_PROB_IF_STRESSED:
                state[b] = 2
            else:
                state[b] = 1

        else:  # cur == 0, healthy — may transition into stress
            p = hazard[b]
            p += CONTAGION_BETA * neighbor_frac * hazard[b]
            if region_shock_active[r]:
                p += REGIONAL_SHOCK_BOOST * hazard[b]
            p = min(p, 0.9)
            if np.random.rand() < p:
                state[b] = 1
                if region_shock_active[r] and neighbor_frac > 0:
                    reason[b] = "Independent Regional Shock"
                elif neighbor_frac > 0 and np.random.rand() < (CONTAGION_BETA * neighbor_frac * hazard[b]) / p:
                    reason[b] = "Contagion-Driven Stress"
                elif region_shock_active[r]:
                    reason[b] = "Independent Regional Shock"
                else:
                    reason[b] = "Isolated Stress"
            else:
                state[b] = 0

        records.append({
            "borrower_id": b, "t": t,
            "state": state[b],
            "is_stressed": 1 if state[b] in (1, 2) else 0,
            "own_lagged_risk": 1 if cur in (1, 2) else 0,
            "neighbor_stress_fraction": neighbor_frac,
            "regional_shock_index": int(region_shock_active[r]),
            "ground_truth_reason": reason[b],
        })

panel_df = pd.DataFrame(records)
panel_df = panel_df.merge(
    borrowers_df[["borrower_id", "group_id", "partner_id", "region_id",
                   "loan_amount", "term_months", "repayment_interval", "current_savings"]],
    on="borrower_id", how="left"
)

print(f"OUTPUT: panel_df — shape={panel_df.shape} "
      f"({panel_df['borrower_id'].nunique()} borrowers x {T} months)")
print("Final-month stress rate (state 1 or 2):", panel_df[panel_df.t == T-1]['is_stressed'].mean().round(3))
print("Ever-defaulted rate:", (panel_df.groupby('borrower_id')['state'].max() == 2).mean().round(3))
print(panel_df['ground_truth_reason'].value_counts(dropna=False))

OUTPUT: panel_df — shape=(24000, 15) (2000 borrowers x 12 months)
Final-month stress rate (state 1 or 2): 0.271
Ever-defaulted rate: 0.11
ground_truth_reason
None                          19433
Isolated Stress                2832
Contagion-Driven Stress         880
Independent Regional Shock      855
Name: count, dtype: int64


In [7]:
# ============================================================
# NEW CELL 6B — Shared borrower-level train/test split
# Same split reused by Model 1, Model 3, and Model 2 (in Notebook B) so
# no model leaks a borrower's other months across train/test, and all
# three models' held-out results are directly comparable.
# Output: `train_borrowers`, `test_borrowers` sets; `borrower_split.csv` saved
# ============================================================
unique_borrowers = borrowers_df["borrower_id"].unique()
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(unique_borrowers)
split_idx = int(0.8 * len(unique_borrowers))
train_borrowers = set(unique_borrowers[:split_idx])
test_borrowers  = set(unique_borrowers[split_idx:])

split_df = pd.DataFrame({
    "borrower_id": list(train_borrowers) + list(test_borrowers),
    "split": ["train"] * len(train_borrowers) + ["test"] * len(test_borrowers),
})
split_df.to_csv(OUTPUT_DIR + "borrower_split.csv", index=False)

print(f"OUTPUT: borrower_split.csv saved — {len(train_borrowers)} train / {len(test_borrowers)} test borrowers")

OUTPUT: borrower_split.csv saved — 1600 train / 400 test borrowers


In [8]:
# ============================================================
# CELL 7 — Model 1: Individual Borrower Risk (XGBoost, GPU-accelerated)
# FIXED: filtered to own_lagged_risk == 0 (genuine new onset only, mirrors
# Model 3), split by BORROWER not by row (uses Cell 6B's shared split, no
# leakage), reports held-out test AUC instead of train AUC. own_lagged_risk
# dropped from features post-filter (constant 0, dead weight); `t` added
# instead since hazard may have a natural shape over the loan term.
# Output: `xgboost_individual.json` saved; `individual_risk_score` written
#          back onto every row of panel_df (used by the combination layer)
# ============================================================
from sklearn.metrics import roc_auc_score

feature_cols_m1 = ["loan_amount", "term_months", "current_savings", "t"]
panel_encoded = pd.get_dummies(panel_df, columns=["repayment_interval"], prefix="repay")
repay_cols = [c for c in panel_encoded.columns if c.startswith("repay_")]
feature_cols_m1 += repay_cols

train_df = panel_encoded[panel_encoded["own_lagged_risk"] == 0].copy()
train_mask = train_df["borrower_id"].isin(train_borrowers)
test_mask  = train_df["borrower_id"].isin(test_borrowers)

X_train, y_train = train_df.loc[train_mask, feature_cols_m1], train_df.loc[train_mask, "is_stressed"]
X_test,  y_test  = train_df.loc[test_mask, feature_cols_m1],  train_df.loc[test_mask, "is_stressed"]

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)

params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",
    "max_depth": 4,
    "eta": 0.1,
    "seed": RANDOM_SEED,
}

model1 = xgb.train(params, dtrain, num_boost_round=150)
model1.save_model(OUTPUT_DIR + "xgboost_individual.json")

test_preds = model1.predict(dtest)

# score EVERY row (not just the filtered/held-out set) so downstream
# combination layer + export have a score for every borrower-timestep
all_dmatrix = xgb.DMatrix(panel_encoded[feature_cols_m1])
panel_df["individual_risk_score"] = model1.predict(all_dmatrix)

print(f"OUTPUT: xgboost_individual.json saved — trained on {X_train.shape[0]} rows "
      f"({len(train_borrowers)} borrowers), {X_train.shape[1]} features")
print(f"Held-out test AUC ({len(test_borrowers)} borrowers, {X_test.shape[0]} rows): "
      f"{roc_auc_score(y_test, test_preds):.3f}")

OUTPUT: xgboost_individual.json saved — trained on 16019 rows (1600 borrowers), 7 features
Held-out test AUC (400 borrowers, 3956 rows): 0.528


In [9]:
# ============================================================
# CELL 8 — Model 3: Causal Explainer (regional-shock-controlled logistic regression)
# FIXED: fit on TRAIN borrowers only (Cell 6B's shared split), agreement
# check now computed on HELD-OUT test borrowers, not in-sample.
# Output: `explainer_model.joblib` + `explainer_model.json` saved
# ============================================================
reg_df = panel_df[panel_df["own_lagged_risk"] == 0].copy()
reg_train = reg_df[reg_df["borrower_id"].isin(train_borrowers)]
reg_test  = reg_df[reg_df["borrower_id"].isin(test_borrowers)]

X_cols = ["neighbor_stress_fraction", "regional_shock_index"]
X_train_m3 = sm.add_constant(reg_train[X_cols].astype(float))
y_train_m3 = reg_train["is_stressed"].astype(float)

explainer_model = sm.Logit(y_train_m3, X_train_m3).fit(disp=False)

neighbor_coef = explainer_model.params["neighbor_stress_fraction"]
neighbor_pval = explainer_model.pvalues["neighbor_stress_fraction"]
regional_coef = explainer_model.params["regional_shock_index"]
regional_pval = explainer_model.pvalues["regional_shock_index"]
SIGNIFICANCE = 0.05

def classify_reason(neighbor_frac, regional_shock):
    if regional_shock == 1 and regional_pval < SIGNIFICANCE:
        return "Independent Regional Shock"
    if neighbor_frac > 0 and neighbor_pval < SIGNIFICANCE:
        return "Contagion-Driven Stress"
    return "Isolated Stress"

joblib.dump({
    "params": explainer_model.params.to_dict(),
    "pvalues": explainer_model.pvalues.to_dict(),
    "significance_threshold": SIGNIFICANCE,
}, OUTPUT_DIR + "explainer_model.joblib")

with open(OUTPUT_DIR + "explainer_model.json", "w") as f:
    json.dump({
        "neighbor_coef": float(neighbor_coef), "neighbor_pval": float(neighbor_pval),
        "regional_coef": float(regional_coef), "regional_pval": float(regional_pval),
    }, f, indent=2)

print("OUTPUT: explainer_model.joblib + explainer_model.json saved")
print(f"neighbor_stress_fraction coef={neighbor_coef:.3f}, p={neighbor_pval:.4f}")
print(f"regional_shock_index coef={regional_coef:.3f}, p={regional_pval:.4f}")

# HELD-OUT sanity check — test borrowers only, not seen during fitting
newly_stressed_test = reg_test[reg_test["is_stressed"] == 1].copy()
newly_stressed_test["predicted_reason"] = newly_stressed_test.apply(
    lambda r: classify_reason(r["neighbor_stress_fraction"], r["regional_shock_index"]), axis=1
)
agreement = (newly_stressed_test["predicted_reason"] == newly_stressed_test["ground_truth_reason"]).mean()
print(f"HELD-OUT sanity check ({len(newly_stressed_test)} test-borrower onset events): "
      f"model-recovered reason matches injected ground truth {agreement:.1%} of the time")
print("NOTE for pitch: this validates the identification strategy recovers a KNOWN "
      "injected mechanism in simulation — it is not a real-world accuracy claim.")

OUTPUT: explainer_model.joblib + explainer_model.json saved
neighbor_stress_fraction coef=1.345, p=0.0000
regional_shock_index coef=1.261, p=0.0000
HELD-OUT sanity check (264 test-borrower onset events): model-recovered reason matches injected ground truth 70.8% of the time
NOTE for pitch: this validates the identification strategy recovers a KNOWN injected mechanism in simulation — it is not a real-world accuracy claim.


In [10]:
# ============================================================
# NEW CELL 8B — Forward Shock & Intervention Simulator
# Uses Model 3's FITTED coefficients in place of Cell 6's hardcoded
# ground-truth CONTAGION_BETA/REGIONAL_SHOCK_BOOST. This is the actual
# Point D/E deliverable — Cell 6 only generates ground truth, it was
# never itself the forward-looking simulator the product needs.
# Output: `simulate_shock()`, `simulate_intervention()` — callable here,
#          later exposed via the FastAPI backend.
# ============================================================
SIM_MONTHS = 6
N_MONTE_CARLO = 200

FITTED_CONTAGION_BETA = max(neighbor_coef, 0.0)
FITTED_REGIONAL_BOOST = max(regional_coef, 0.0)

def _p_stress(own_hazard, neighbor_frac, regional_shock):
    p = own_hazard
    p += FITTED_CONTAGION_BETA * neighbor_frac * own_hazard
    if regional_shock:
        p += FITTED_REGIONAL_BOOST * own_hazard
    return min(p, 0.9)

def _run_once(shocked_borrowers, months=SIM_MONTHS, override_healthy=None):
    """override_healthy: borrower_ids forced to state=0 at t=0 (an intervention)."""
    sim_state = {b: 0 for b in borrowers_df["borrower_id"]}
    for b in shocked_borrowers:
        sim_state[b] = 1
    if override_healthy:
        for b in override_healthy:
            sim_state[b] = 0
    trace = []
    for t in range(months):
        prev = sim_state.copy()
        for b in borrowers_df["borrower_id"]:
            if t == 0 and override_healthy and b in override_healthy:
                continue
            if prev[b] == 2:
                sim_state[b] = 2
            elif prev[b] == 1:
                roll = np.random.rand()
                if roll < RECOVERY_PROB:
                    sim_state[b] = 0
                elif roll < RECOVERY_PROB + DEFAULT_PROB_IF_STRESSED:
                    sim_state[b] = 2
                else:
                    sim_state[b] = 1
            else:
                nbrs = group_neighbors.get(b, [])
                nfrac = np.mean([1 if prev[n] in (1, 2) else 0 for n in nbrs]) if nbrs else 0.0
                rshock = int(region_shock_active.get(regions[b], False))
                p = _p_stress(hazard[b], nfrac, rshock)
                sim_state[b] = 1 if np.random.rand() < p else 0
        trace.append(dict(sim_state))
    return trace

def simulate_shock(borrower_id, months=SIM_MONTHS, n_runs=N_MONTE_CARLO):
    """Monte Carlo: shock one borrower, see who follows and roughly when."""
    follow_month = {b: [] for b in borrowers_df["borrower_id"]}
    for _ in range(n_runs):
        trace = _run_once(shocked_borrowers=[borrower_id], months=months)
        already = {borrower_id}
        for t, snapshot in enumerate(trace):
            for b, s in snapshot.items():
                if s in (1, 2) and b not in already:
                    follow_month[b].append(t)
                    already.add(b)
    rows = [
        {"borrower_id": b, "p_follows": len(ml) / n_runs, "median_lag_months": float(np.median(ml)) + 1}
        for b, ml in follow_month.items() if b != borrower_id and ml
    ]
    return pd.DataFrame(rows).sort_values("p_follows", ascending=False)

def simulate_intervention(borrower_id, months=SIM_MONTHS, n_runs=N_MONTE_CARLO):
    """Compare downstream group default count WITH vs WITHOUT intervening on borrower_id."""
    group_id = borrowers_df.set_index("borrower_id").loc[borrower_id, "group_id"]
    group_members = borrowers_df[borrowers_df["group_id"] == group_id]["borrower_id"].tolist()

    without_d, with_d = [], []
    for _ in range(n_runs):
        trace_a = _run_once(shocked_borrowers=[borrower_id], months=months)
        without_d.append(sum(1 for b in group_members if trace_a[-1][b] == 2))
        trace_b = _run_once(shocked_borrowers=[borrower_id], months=months, override_healthy={borrower_id})
        with_d.append(sum(1 for b in group_members if trace_b[-1][b] == 2))

    return {
        "group_id": group_id, "group_size": len(group_members),
        "avg_defaults_without_intervention": float(np.mean(without_d)),
        "avg_defaults_with_intervention": float(np.mean(with_d)),
        "risk_reduction": float(np.mean(without_d) - np.mean(with_d)),
    }

demo_borrower = borrowers_df["borrower_id"].iloc[0]
shock_result = simulate_shock(demo_borrower)
intervention_result = simulate_intervention(demo_borrower)

print(f"OUTPUT: simulate_shock() / simulate_intervention() defined using FITTED "
      f"coefficients (neighbor_beta={FITTED_CONTAGION_BETA:.3f}, regional_beta={FITTED_REGIONAL_BOOST:.3f})")
print(f"\nDemo — shock on {demo_borrower}, top predicted followers:\n{shock_result.head()}")
print(f"\nDemo — intervention on {demo_borrower}: {intervention_result}")

OUTPUT: simulate_shock() / simulate_intervention() defined using FITTED coefficients (neighbor_beta=1.345, regional_beta=1.261)

Demo — shock on B0000, top predicted followers:
     borrower_id  p_follows  median_lag_months
16         B0017      0.655                3.0
911        B0912      0.655                3.0
1943       B1944      0.625                3.0
681        B0682      0.605                3.0
1550       B1551      0.605                3.0

Demo — intervention on B0000: {'group_id': 'G0199', 'group_size': 5, 'avg_defaults_without_intervention': 0.375, 'avg_defaults_with_intervention': 0.205, 'risk_reduction': 0.17}


In [11]:
# ============================================================
# NEW CELL 8C — Combination layer: one payload per borrower-timestep
# Combines Model 1 (individual score) + Model 3 (causal reason) into the
# single object structure the FastAPI backend needs to serve to Flutter.
# Model 2's (GraphSAGE, Notebook B) exposure score is left as a
# placeholder — joined in during the final reconciliation step once
# Notebook B has run, not fabricated here.
# Output: `combined_scores.csv` saved
# ============================================================
combined = panel_df[panel_df["own_lagged_risk"] == 0].copy()
combined["predicted_reason"] = combined.apply(
    lambda r: classify_reason(r["neighbor_stress_fraction"], r["regional_shock_index"])
    if r["is_stressed"] == 1 else "Healthy",
    axis=1
)
combined["confidence"] = combined.apply(
    lambda r: neighbor_pval if r["predicted_reason"] == "Contagion-Driven Stress"
    else (regional_pval if r["predicted_reason"] == "Independent Regional Shock" else np.nan),
    axis=1
)
combined["multi_hop_exposure_score"] = np.nan  # placeholder — filled from Notebook B's node_embeddings.csv

output_cols = ["borrower_id", "t", "group_id", "region_id", "individual_risk_score",
               "neighbor_stress_fraction", "regional_shock_index",
               "predicted_reason", "confidence", "multi_hop_exposure_score", "is_stressed"]
combined[output_cols].to_csv(OUTPUT_DIR + "combined_scores.csv", index=False)

print(f"OUTPUT: combined_scores.csv saved — shape={combined[output_cols].shape}")
print(combined["predicted_reason"].value_counts())

OUTPUT: combined_scores.csv saved — shape=(19975, 11)
predicted_reason
Healthy                       18662
Contagion-Driven Stress         671
Isolated Stress                 399
Independent Regional Shock      243
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 9 — Export the demo dataset + handoff files for the GraphSAGE notebook
# FIXED: previously only sampled t == T-1, which mechanically inflated
# apparent contagion — by the final month almost every new transition has
# SOME stressed neighbor nearby just from elapsed time. Now takes each
# borrower's FIRST transition into stress (any month), so the exported
# reason distribution reflects the full panel's true ratio instead of an
# inverted one.
# ============================================================
first_transition = (
    panel_df[panel_df["is_stressed"] == 1]
    .sort_values("t")
    .groupby("borrower_id", as_index=False)
    .first()
)
never_stressed_final = (
    panel_df[~panel_df["borrower_id"].isin(first_transition["borrower_id"])]
    .sort_values("t")
    .groupby("borrower_id", as_index=False)
    .last()
)
snapshot_df = pd.concat([first_transition, never_stressed_final], ignore_index=True)

target_groups = np.random.choice(
    snapshot_df["group_id"].unique(),
    size=min(18, snapshot_df["group_id"].nunique()),
    replace=False
)
sample_df = snapshot_df[snapshot_df["group_id"].isin(target_groups)].copy()

if len(sample_df) > 150:
    keep_groups, running_total = [], 0
    for g in target_groups:
        g_size = (sample_df["group_id"] == g).sum()
        if running_total + g_size > 150:
            break
        keep_groups.append(g)
        running_total += g_size
    sample_df = sample_df[sample_df["group_id"].isin(keep_groups)]

sample_df["stress_reason"] = sample_df.apply(
    lambda r: classify_reason(r["neighbor_stress_fraction"], r["regional_shock_index"])
    if r["is_stressed"] == 1 else "Not Stressed",
    axis=1
)

export_cols = ["borrower_id", "group_id", "partner_id", "region_id", "loan_amount",
               "term_months", "repayment_interval", "current_savings", "is_stressed", "stress_reason"]
real_kiva_sample = sample_df[export_cols].reset_index(drop=True)
real_kiva_sample.to_csv(OUTPUT_DIR + "real_kiva_sample.csv", index=False)

panel_df.to_csv(OUTPUT_DIR + "full_panel.csv", index=False)
borrowers_df.to_csv(OUTPUT_DIR + "node_features.csv", index=False)

print(f"OUTPUT: real_kiva_sample.csv saved — {len(real_kiva_sample)} borrowers, "
      f"{real_kiva_sample['group_id'].nunique()} groups")
print(real_kiva_sample["stress_reason"].value_counts())
print("\nFull-panel reference ratio (should roughly match the sample above now):")
print(panel_df[panel_df["is_stressed"] == 1]["ground_truth_reason"].value_counts(normalize=True).round(2))
print("\nOUTPUT: full_panel.csv + node_features.csv also saved.")
print("Next: click 'Save Version' → publish this notebook's output as a")
print("Kaggle Dataset → attach it as an input to Notebook B under Account 2.")

OUTPUT: real_kiva_sample.csv saved — 101 borrowers, 18 groups
stress_reason
Not Stressed                  44
Contagion-Driven Stress       23
Isolated Stress               23
Independent Regional Shock    11
Name: count, dtype: int64

Full-panel reference ratio (should roughly match the sample above now):
ground_truth_reason
Isolated Stress               0.62
Contagion-Driven Stress       0.19
Independent Regional Shock    0.19
Name: proportion, dtype: float64

OUTPUT: full_panel.csv + node_features.csv also saved.
Next: click 'Save Version' → publish this notebook's output as a
Kaggle Dataset → attach it as an input to Notebook B under Account 2.


In [13]:
# ============================================================
# CELL 10 — Confirm all Notebook A artifacts exist
# ============================================================
for f in ["xgboost_individual.json", "explainer_model.joblib", "explainer_model.json",
          "real_kiva_sample.csv", "edge_list.csv", "group_edge_list.csv", "node_features.csv",
          "full_panel.csv", "borrower_split.csv", "combined_scores.csv"]:
    path = OUTPUT_DIR + f
    print(f"[{'OK' if os.path.exists(path) else 'MISSING'}] {f}")

[OK] xgboost_individual.json
[OK] explainer_model.joblib
[OK] explainer_model.json
[OK] real_kiva_sample.csv
[OK] edge_list.csv
[OK] group_edge_list.csv
[OK] node_features.csv
[OK] full_panel.csv
[OK] borrower_split.csv
[OK] combined_scores.csv
